In [1]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

data_root = "data/7k"
for dirname, _, filenames in os.walk(data_root):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        break

data/7k/json_dataset.json
data/7k/image/491.png
data/7k/json/60710.json


In [2]:
import json
import pandas as pd
import glob
from tqdm import tqdm

# Path to JSON folder
json_path = data_root + "/json/*.json"

file_paths = glob.glob(json_path)

print(f"Found {len(file_paths)} JSON files.")

rows = []

for file in tqdm(file_paths):
    try:
        with open(file, "r") as f:
            data = json.load(f)

        row = {
            "file_name": file.split("/")[-1],
            "invoice_number": data.get("invoice", {}).get("number"),
            "invoice_date": data.get("invoice", {}).get("date"),
            "buyer_name": data.get("buyer", {}).get("name"),
            "buyer_country": data.get("buyer", {}).get("country"),
            "seller_company": data.get("seller", {}).get("company_name"),
            "seller_country": data.get("seller", {}).get("country"),
            "vat_number": data.get("seller", {}).get("vat_no"),
            "sub_total": data.get("payment", {}).get("sub_total"),
            "total_amount": data.get("payment", {}).get("total"),
            "discount_amount": data.get("payment", {}).get("discount_amount"),
            "discount_percentage": data.get("payment", {}).get("discount_percentage"),
            "currency": data.get("payment", {}).get("currency"),
        }

        # Combine product descriptions into one text field
        descriptions = []
        if "products" in data:
            for p in data["products"]:
                descriptions.append(p.get("description", ""))

        row["invoice_text"] = " ".join(descriptions)

        rows.append(row)

    except Exception as e:
        print(f"Error reading {file}: {e}")

df = pd.DataFrame(rows)
df.head()

Found 7000 JSON files.


100%|██████████| 7000/7000 [00:00<00:00, 34529.91it/s]


,file_name,invoice_number,invoice_date,buyer_name,buyer_country,seller_company,seller_country,vat_number,sub_total,total_amount,discount_amount,discount_percentage,currency,invoice_text
0,60710.json,1208-2643,07.07.1990,None,None,None,None,None,NaN,197.21,NaN,NaN,None,disintermediate integrated experiences redefin...
1,50373.json,276044,23.07.1994,None,None,None,None,None,NaN,NaN,NaN,NaN,None,e-enable back-end paradigms scale intuitive so...
2,60934.json,7827-5869,19.04.2007,None,None,None,None,None,NaN,110.21,NaN,NaN,None,expedite out-of-the-box web-readiness mesh ext...
3,10928.json,5895-9419,03.12.2012,None,None,None,None,None,NaN,509.11,NaN,NaN,None,generate web-enabled markets
4,50505.json,727213,17.01.1992,None,None,None,None,None,NaN,NaN,NaN,NaN,None,aggregate visionary networks


In [3]:
def print_details(file_name):
    with open(data_root + f"/json/{file_name}.json") as f:
        truth = json.load(f)

        try:
            contact = "\n".join(truth["buyer"].values())
            invoice_no  = str(truth["invoice"]["number"])

            total = 0
            if "payment" in truth.keys() and "total" in truth["payment"].keys():
                total = truth["payment"]["total"]
            elif "payment" in truth.keys() and "account_number" in truth["payment"].keys():
                total = "\n".join([str(x["unit_price"]) for x in truth["products"]])
            elif "tax" in truth.keys() and "amount_0" in truth["tax"].keys():
                total = truth["tax"]["amount_0"]
            elif "tax" in truth.keys() and "amount_0" in truth["tax"].keys():
                total = truth["tax"]["amount_0"]
            elif "tax" in truth.keys() and "amount_excluding_tax" in truth["tax"].keys():
                total = truth["tax"]["amount_excluding_tax"]

            print(truth)

            return dict(invoice_no=invoice_no, total=total, contact=contact)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")

In [4]:
print_details("001")

{'buyer': {'address': '960 Hurley Springs North Alyssa, RI 49322', 'company_name': 'Hayes LLC', 'country': 'Lebanon', 'name': 'Mercedes Martinez'}, 'invoice': {'date': '05.08.2007', 'number': 802205}, 'payment': {'discount_amount': 43.03, 'discount_percentage': 11.77, 'due': 61.79, 'sub_total': 141.66, 'total': 534.11}, 'products': [{'description': 'transform open-source info-mediaries', 'index': '10', 'quantity': 6.27, 'total_price': 329.62, 'unit_price': 41.57}, {'description': 'innovate transparent e-services', 'index': '9', 'quantity': 5.62, 'total_price': 986.28, 'unit_price': 10.47}, {'description': 'optimize innovative action-items', 'index': '3', 'quantity': 7.11, 'total_price': 559.69, 'unit_price': 97.41}, {'description': 'enhance e-business initiatives', 'index': '10', 'quantity': 4.75, 'total_price': 311.4, 'unit_price': 88.26}, {'description': 'morph value-added markets', 'index': '2', 'quantity': 8.39, 'total_price': 831.07, 'unit_price': 65.07}, {'description': 'cultivat

{'invoice_no': '802205',
 'total': 534.11,
 'contact': '960 Hurley Springs North Alyssa, RI 49322\nHayes LLC\nLebanon\nMercedes Martinez'}

In [5]:
print_details("10001")

{'buyer': {'address': '13202 Crystal Walk Apt. 302 North Leslie, TN 10464', 'email': 'calvin51@example.com', 'phone': '925-915-3160x4029', 'vat_no': 'Js76655417325'}, 'invoice': {'date': '03.08.2018', 'due_date': '09.04.2004', 'market_reference': 717165, 'notes': 'mesh holistic partnerships', 'number': '5875-8927', 'reference': 349586, 'vat_no': 'Xg79283353388'}, 'payment': {'net_total': 171.86, 'total': 809.58}, 'products': [{'amount': 426.47, 'description': 'benchmark best-of-breed interfaces', 'hours': 3.94, 'rate': 9.63}, {'amount': 268.79, 'description': 'brand plug-and-play infrastructures', 'hours': 9.93, 'rate': 5.59}, {'amount': 371.45, 'description': 'aggregate killer models', 'hours': 4.39, 'rate': 4.98}, {'amount': 350.53, 'description': 'enable dynamic relationships', 'hours': 1.23, 'rate': 8.05}, {'amount': 108.15, 'description': 'deliver enterprise web-readiness', 'hours': 9.28, 'rate': 3.98}, {'amount': 447.23, 'description': 're-contextualize front-end relationships', 

{'invoice_no': '5875-8927',
 'total': 809.58,
 'contact': '13202 Crystal Walk Apt. 302 North Leslie, TN 10464\ncalvin51@example.com\n925-915-3160x4029\nJs76655417325'}

In [6]:
print_details("20001")

{'buyer': {'address': 'PSC 5909, Box 3621 APO AP 88691', 'company_name': 'Reed Inc', 'email': 'awilliams@example.org', 'name': 'Jamie Bennett', 'phone': '001-856-932-8020'}, 'invoice': {'date': '15.11.1993', 'due_date': '07.12.2010', 'number': 'Mct65061999539'}, 'payment': {'account_number': 3628988967, 'bank_name': 'Hart, Walker and Cox', 'routing_number': 195818226}, 'products': [{'amount': 80.89, 'description': 'morph out-of-the-box synergies', 'index': '8', 'quantity': 4.21, 'unit_price': 89.5}, {'amount': 217.54, 'description': 'incubate mission-critical models', 'index': '10', 'quantity': 1.63, 'unit_price': 33.94}, {'amount': 446.99, 'description': 'reinvent e-business platforms', 'index': '4', 'quantity': 4.87, 'unit_price': 87.87}, {'amount': 306.79, 'description': 'enable 24/7 methodologies', 'index': '6', 'quantity': 5.17, 'unit_price': 58.57}, {'amount': 254.68, 'description': 'embrace integrated vortals', 'index': '3', 'quantity': 2.81, 'unit_price': 52.4}, {'amount': 37.4

{'invoice_no': 'Mct65061999539',
 'total': '89.5\n33.94\n87.87\n58.57\n52.4\n55.8\n15.8',
 'contact': 'PSC 5909, Box 3621 APO AP 88691\nReed Inc\nawilliams@example.org\nJamie Bennett\n001-856-932-8020'}

In [7]:
print_details("30001")

{'buyer': {'address': '21891 Benson Hill Apt. 514 Christopherton, HI 27587', 'company_name': 'Duarte-Martinez', 'country': 'Brazil', 'email': 'ysimon@example.net', 'name': 'Kathleen Fernandez', 'phone': '230.402.4655x9030', 'vat_no': 'le15496524343'}, 'invoice': {'date': '12.11.2021', 'due_date': '24.11.2023', 'linked_invoice': 'yv-6749-5501', 'notes': 'deploy sticky supply-chains', 'number': 'CP-0988-7745'}, 'products': [{'description': 'benchmark granular infrastructures', 'discount_percentage': 8.32, 'price': 442.41, 'quantity': 7.82, 'unit_price': 14.99}, {'description': 'iterate granular content', 'discount_percentage': 19.95, 'price': 197.16, 'quantity': 5.23, 'unit_price': 66.99}, {'description': 'drive impactful action-items', 'discount_percentage': 7.69, 'price': 394.79, 'quantity': 1.15, 'unit_price': 90.15}, {'description': 'repurpose innovative e-services', 'discount_percentage': 10.52, 'price': 129.43, 'quantity': 5.28, 'unit_price': 86.53}, {'description': 'leverage brick

{'invoice_no': 'CP-0988-7745',
 'total': 23.81,
 'contact': '21891 Benson Hill Apt. 514 Christopherton, HI 27587\nDuarte-Martinez\nBrazil\nysimon@example.net\nKathleen Fernandez\n230.402.4655x9030\nle15496524343'}

In [8]:
print_details("40001")

{'buyer': {'address': '26255 Stevenson Walk New William, NE 74949'}, 'invoice': {'bc_no': 'OP38391', 'date': '10.05.2018', 'maturity_date': '21.06.2004', 'number': 840223}, 'products': [{'amount': 222.32, 'description': 'revolutionize bleeding-edge action-items', 'quantity': 1.87, 'unit_price': 71.3, 'vat_amount': 2.04}, {'amount': 396.91, 'description': 'visualize cross-platform partnerships', 'quantity': 2.88, 'unit_price': 38.76, 'vat_amount': 7.94}, {'amount': 48.18, 'description': 're-intermediate clicks-and-mortar networks', 'quantity': 6.0, 'unit_price': 90.26, 'vat_amount': 14.75}, {'amount': 282.63, 'description': 'embrace e-business interfaces', 'quantity': 5.01, 'unit_price': 97.88, 'vat_amount': 13.19}, {'amount': 255.89, 'description': 'utilize best-of-breed methodologies', 'quantity': 1.72, 'unit_price': 46.45, 'vat_amount': 7.36}], 'seller': {'address': '95468 Matthew Springs Apt. 341 Clayland, MA 08502'}, 'tax': {'amount_excluding_tax': 72.01, 'amount_including_tax': 67

{'invoice_no': '840223',
 'total': 72.01,
 'contact': '26255 Stevenson Walk New William, NE 74949'}

In [9]:
print_details("50001")

{'buyer': {'address': '60304 Crawford Trace Apt. 069 New Aaron, ID 76969', 'id': 'uP730'}, 'invoice': {'bc_no': 'TW27540', 'currency_code': 'IMP', 'date': '09.05.2014', 'due_date': '13.05.1992', 'number': 829839}, 'payment': {'bank_name': 'Coleman-Watkins', 'bic': 'SNOMGB60EU4', 'iban': 'UDYJ63951388880527'}, 'products': [{'amount': 24.12, 'description': 'enable distributed action-items', 'index': '10', 'price': 427.99, 'quantity': 1.05, 'tax_amount': 61.17, 'total_ttc': 346.91, 'vat_amount': 4.07}, {'amount': 487.72, 'description': 'seize integrated mindshare', 'index': '5', 'price': 115.8, 'quantity': 3.34, 'tax_amount': 96.9, 'total_ttc': 301.52, 'vat_amount': 6.36}], 'seller': {'name': 'Jennifer Williams'}, 'supplier': {'address': '93059 Mercado Canyon Suite 069 Williamston, MO 59416', 'name': 'Kiara Parks'}, 'tax': {'amount_excluding_tax': 52.98, 'amount_including_tax': 168.96, 'tax_amount': 10.3, 'total_ht': 265.33, 'total_ttc': 162.93}}


{'invoice_no': '829839',
 'total': 52.98,
 'contact': '60304 Crawford Trace Apt. 069 New Aaron, ID 76969\nuP730'}

In [10]:
print_details("60001")

{'buyer': {'address': '62765 Nichols Road Suite 063 Jenniferside, WV 35221', 'business_no': 'Rr426', 'id': 'DC532'}, 'invoice': {'date': '15.03.2014', 'number': '8841-1279'}, 'payment': {'due_date': '03.09.1993', 'total': 887.01}, 'products': [{'amount': 434.17, 'description': 'aggregate back-end niches', 'quantity': 5.44, 'reference': 'SE-832', 'unit_price': 72.97}, {'amount': 401.23, 'description': 'streamline seamless networks', 'quantity': 2.83, 'reference': 'Lo-643', 'unit_price': 14.45}, {'amount': 196.31, 'description': 'maximize back-end applications', 'quantity': 7.23, 'reference': 'PY-816', 'unit_price': 83.9}, {'amount': 228.47, 'description': 'redefine magnetic users', 'quantity': 4.49, 'reference': 'sS-857', 'unit_price': 62.38}, {'amount': 253.55, 'description': 'implement 24/7 methodologies', 'quantity': 2.0, 'reference': 'cR-268', 'unit_price': 24.85}, {'amount': 374.66, 'description': 'revolutionize rich models', 'quantity': 1.09, 'reference': 'aX-190', 'unit_price': 4

{'invoice_no': '8841-1279',
 'total': 887.01,
 'contact': '62765 Nichols Road Suite 063 Jenniferside, WV 35221\nRr426\nDC532'}